# TP2 — Structured Outputs with LangChain and Mistral

## Objectives
- Understand why structured outputs make LLM responses reliable.
- Define Pydantic schemas and use `with_structured_output`.
- Explore output formats: boolean, enum action, integer score, translation, step-by-step reasoning.


## Setup — load API key


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
assert os.getenv('MISTRAL_API_KEY'), 'Missing MISTRAL_API_KEY in .env'
print('API key loaded.')


## Step 2 — Boolean Structured Output (Ex 2.1, 2.2)


In [ ]:
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai.chat_models import ChatMistralAI

class Answer(BaseModel):
    answer: bool

# Ex 2.1: strengthened prompt handles ambiguous questions
prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'You are an assistant that answers only True or False to the user\'s question. '
        'If the question is ambiguous, answer based on the most literal interpretation. '
        'Provide no explanation, only the boolean value.'
    )),
    ('human', '{question}'),
])

llm = ChatMistralAI(model='mistral-large-latest', temperature=0)
chain = prompt | llm.with_structured_output(schema=Answer)

def answer_question(question: str) -> bool:
    return chain.invoke({'question': question}).answer

questions = [
    'Christmas is in winter',
    'It rains when it does not rain',
    'A square is a rectangle',  # Ex 2.1 — ambiguous
]

for q in questions:
    resp = answer_question(q)
    # Ex 2.2: type is bool — usable directly in if/else application logic
    print(f'Q: {q}')
    print(f'A: {resp}  (type: {type(resp).__name__})')
    print()


## Step 3 — Predefined Action Selection (Ex 3.1, 3.2)


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai.chat_models import ChatMistralAI

# Ex 3.1: added third action
tasks = [
    'Answer a new question',
    'Provide more details about the previous question',
    'Ask for clarification',
]

class NextTask(BaseModel):
    """Always use this tool to structure your response."""
    action: str = Field(..., enum=tasks, description='The next action to take')

prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'Classify the user request into one of the available actions. '
        "If the intent is unclear, choose 'Ask for clarification'."
    )),
    ('human', '{text}'),
])

llm = ChatMistralAI(model='mistral-large-latest', temperature=0)
chain = prompt | llm.with_structured_output(schema=NextTask)

messages = [
    'Can you tell me more?',
    'What are PPVs?',
    "I'm not sure... it's something I saw somewhere",  # Ex 3.1
]

for text in messages:
    result = chain.invoke({'text': text})
    print(f'Message: {text}')
    print(f'Action : {result.action}')
    print()

# Ex 3.2: structured output prevents hallucinated actions
print('Ex 3.2: the enum constraint makes out-of-list values impossible — Pydantic raises ValidationError.')


## Step 4 — Integer Score + Comment + Aggregated Average (Ex 4.1, 4.2)


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai.chat_models import ChatMistralAI

# Ex 4.1: added comment field
class MessageTone(BaseModel):
    tone_score: int = Field(..., ge=1, le=5, description='1=neutral/cold, 5=very friendly')
    comment: str = Field(..., description='1-2 sentence justification for the score')

prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'Evaluate the tone of the user message. '
        'Assign a score 1-5 (1=neutral/cold, 5=very friendly). '
        'Provide a short comment justifying the score.'
    )),
    ('human', '{text}'),
])

llm = ChatMistralAI(model='mistral-large-latest', temperature=0)
chain = prompt | llm.with_structured_output(schema=MessageTone)

messages = [
    'Hello, could you help me please?',
    'I need this immediately.',
    'Thank you so much for your precious help!',
    'What is this bug again?',
    'Great work, truly impressive!',
]

scores = []
for text in messages:
    result = chain.invoke({'text': text})
    scores.append(result.tone_score)
    print(f'Message : {text}')
    print(f'Score   : {result.tone_score}/5')
    print(f'Comment : {result.comment}')
    print()

# Ex 4.2: aggregated average
print(f'Average tone score: {sum(scores)/len(scores):.2f} / 5')


## Step 5 — Translation with Structured JSON Output (Ex 5.1, 5.2)


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_mistralai.chat_models import ChatMistralAI

SUPPORTED_LANGUAGES = {
    'english', 'french', 'spanish', 'german', 'italian',
    'portuguese', 'dutch', 'arabic', 'chinese', 'japanese',
}

class Translation(BaseModel):
    original_text: str = Field(..., description='The original text before translation')
    original_language: str = Field(..., description='Language of the original text')
    translated_text: str = Field(..., description='The translated text')
    translated_language: str = Field(..., description='The target language')

def translate(text, source_language='french', target_language='english', return_full_object=False):
    # Ex 5.2: validate languages
    if target_language.lower() not in SUPPORTED_LANGUAGES:
        raise ValueError(f'Unsupported target language: {target_language}')
    if source_language.lower() not in SUPPORTED_LANGUAGES:
        raise ValueError(f'Unsupported source language: {source_language}')

    llm = ChatMistralAI(model='mistral-medium-latest')
    prompt = ChatPromptTemplate.from_template(
        'Translate the text from {source_language} to {target_language}.\n\nText:\n{text}'
    )
    structured_chain = prompt | llm.with_structured_output(Translation)

    # Ex 5.1: return full object or just translated text
    if return_full_object:
        return structured_chain.invoke({'source_language': source_language,
                                        'target_language': target_language, 'text': text})
    chain = structured_chain | RunnableLambda(lambda t: t.translated_text)
    return chain.invoke({'source_language': source_language,
                         'target_language': target_language, 'text': text})

# Basic usage
print(translate('What is the capital of Albania?', source_language='english', target_language='french'))

# Ex 5.1: full object
full = translate('What is the capital of Albania?', source_language='english',
                 target_language='french', return_full_object=True)
print(f'Original: {full.original_text}')
print(f'Translated: {full.translated_text}')

# Ex 5.2: unsupported language raises ValueError
try:
    translate('Hello', source_language='english', target_language='klingon')
except ValueError as e:
    print(f'ValueError: {e}')


## Step 6 — Step-by-Step Math Reasoning (Ex 6.1, 6.2)


In [ ]:
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai.chat_models import ChatMistralAI

class Step(BaseModel):
    explanation: str
    output: str

# Ex 6.1: added verifications field
class MathResponse(BaseModel):
    steps: list[Step]
    final_answer: str
    verifications: list[str]  # Ex 6.1

prompt = ChatPromptTemplate.from_messages([
    ('system', (
        'You are a pedagogical math teacher. Show each step clearly. '
        'After the final answer, list verifications that confirm the solution.'
    )),
    ('human', '{exercise}'),
])

llm = ChatMistralAI(model='mistral-large-latest', temperature=0)
chain = prompt | llm.with_structured_output(schema=MathResponse)

# Ex 6.2: test on different equation types
exercises = [
    'Solve: 8x + 31 = 2',
    'Solve: 3x\u00b2 - 12 = 0',    # non-linear
    'Solve: x/2 + 3/4 = 7/4', # fractions
]

for ex in exercises:
    result = chain.invoke({'exercise': ex})
    print(f'Exercise: {ex}')
    for step in result.steps:
        print(f'  \u2022 {step.explanation} \u2192 {step.output}')
    print(f'Answer: {result.final_answer}')
    print('Verifications:')
    for v in result.verifications:
        print(f'  \u2713 {v}')
    print()
